In [16]:
import os

import evaluate
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

from collator import DataCollatorSpeechSeq2SeqWithPadding

In [17]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [18]:
CSV_PATH = 'data/manifest.csv'
AUDIO_DIR = 'data/0'
MODEL_NAME = 'openai/whisper-tiny'
OUTPUT_DIR = './whisper-small-ru'
N_SAMPLES = 10000

In [19]:
# manifest.csv has unquoted commas in the text column, so split each line on the first comma only
with open(CSV_PATH, encoding='utf-8') as f:
    next(f)
    rows = [line.rstrip('\n').split(',', 1) for line in f]
df = pd.DataFrame(rows, columns=['filename', 'text'])
df = df[['filename', 'text']].dropna()
df['filename'] = df['filename'].apply(lambda x: os.path.join(AUDIO_DIR, x))
df = df[df['filename'].apply(os.path.exists)]
df = df.sample(n=min(N_SAMPLES, len(df)), random_state=42).reset_index(drop=True)
len(df)

10000

In [20]:
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language='russian', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.generation_config.language = 'russian'
model.generation_config.task = 'transcribe'
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False

Loading weights: 100%|██████████| 167/167 [00:00<00:00, 5504.94it/s]


In [21]:
train_df, valid_df = train_test_split(df, test_size=0.1, random_state=42)

atc_dataset_train = Dataset.from_dict(train_df.to_dict(orient='list'))
atc_dataset_valid = Dataset.from_dict(valid_df.to_dict(orient='list')).select(range(min(500, len(valid_df))))

In [22]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [23]:
wer_metric = evaluate.load('wer')


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    return {'wer': 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

In [24]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    fp16=True,
    learning_rate=1e-5,
    warmup_steps=200,
    max_steps=2000,
    eval_strategy='steps',
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    remove_unused_columns=False,
    dataloader_num_workers=4,
    dataloader_persistent_workers=True,
    report_to='none',
)

In [25]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=atc_dataset_train,
    eval_dataset=atc_dataset_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

[W1019 23:53:51.977514744 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 79691776 bytes (free: 102629376, total: 6132858880).


In [26]:
trainer.train()

Step,Training Loss,Validation Loss,Wer
500,0.589549,0.570744,45.363863
1000,0.493817,0.523504,41.283156
1500,0.399288,0.510025,41.623215


[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Writing model shards: 100%

KeyboardInterrupt: 

In [ ]:
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)